# Chains

In [4]:
import google.generativeai as genai

#os_key Setting

from dotenv import load_dotenv
import os
# .env 파일 로드(api key load) 
#GOOGLE_API_KEY 
#LANGCHAIN_API_KEY 
#TAVILY_API_KEY 
#HUGGINGFACEHUB_API_TOKEN 
#COHERE_API_KEY 

load_dotenv()

True

In [2]:
#openai case

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser

In [28]:
llm = ChatOpenAI(model="gpt-4-turbo")

In [3]:
#gemini case

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser

In [6]:
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro")

## Single Chain

In [10]:
# 요약 template

#step 1 : human_message_prompt를 정의한다. 실제 유저 입력인 {text}는 json형태로 invoke로 입력된다.
human_message_prompt = "'{text}' 여기서 키워드를 뽑아서 콤마로 구분해줘"
#step 2 : HumanMessagePromptTemplate.from_template로 human message를 입력한다.
human_message_prompt_template = HumanMessagePromptTemplate.from_template(human_message_prompt)


In [11]:
#system_prompt는 AI에게 별도의 instruction이나 페르소나를 부여할때만 사용되므로 필요하지 않으면 사용하지 않아도 된다.
#step 3 : system_message_prompt를 정의한다. {language}는 json형태로 invoke로 입력된다.
system_message_prompt = "당신은 {language} 언어 전문가다."
#step 4 : SystemMessagePromptTemplate.from_template로 system message를 입력한다.
system_message_prompt_template = SystemMessagePromptTemplate.from_template(system_message_prompt)

In [24]:
#human_message_prompt_template만 정의된 경우우
#step 5 : ChatPromptTemplate.from_messages에 정의된 human_message_prompt_template를 list형태로 입력한다.
chat_prompt_template = ChatPromptTemplate.from_messages([human_message_prompt_template])

In [25]:
chain = chat_prompt_template | llm | StrOutputParser()

In [26]:
out = chain.invoke({"text": "LangChain is a framework for developing applications powered by language models."})

In [27]:
out

'LangChain, framework, language models, applications\n'

In [28]:
#human_message_prompt_template와 system_message_prompt_template 같이 정의된 경우우
#ChatPromptTemplate.from_messages의 list로 system_prmpt와 human_promt 정의하기기
chat_prompt_template = ChatPromptTemplate.from_messages([system_message_prompt_template,human_message_prompt_template])

In [29]:
chain = chat_prompt_template | llm | StrOutputParser()

In [30]:
#error 발생 : {language} 가 정의되지 않았다.
out = chain.invoke({"text": "LangChain is a framework for developing applications powered by language models."})

KeyError: "Input to ChatPromptTemplate is missing variables {'language'}.  Expected: ['language', 'text'] Received: ['text']\nNote: if you intended {language} to be part of the string and not a variable, please escape it with double curly braces like: '{{language}}'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT "

In [31]:
out = chain.invoke({"language" : "영어",
    "text": "LangChain is a framework for developing applications powered by language models."})

In [32]:
out

'Language models, framework, applications\n'

## Multiple chain

In [33]:
# Reference: https://techcrunch.com/2023/11/29/a-timeline-of-sam-altmans-firing-from-openai-and-the-fallout/?guccounter=1&guce_referrer=aHR0cHM6Ly93d3cuZ29vZ2xlLmNvbS8&guce_referrer_sig=AQAAABj9TJeKpfEe-qMrcSB1ZKRYrZ9NitZC84VpXuK3PdufAb1F4VMgQEyoGiLa3fQVRg3zA4CPhQmn-LKO2SsxoDiQchz292_G80ZxNOx34WQ_3Nad0xa_LtwILVKTBgSyjavB3uERdh8SdYbuZ0CSuifL21FfXxjD4Ldv3VlKdm0x
text = """
In a dramatic turn of events late Friday, ex-Y Combinator president Sam Altman was fired as CEO of AI startup OpenAI, the company behind viral AI hits like ChatGPT, GPT-4 and DALL-E 3, by OpenAI’s board of directors. Then, the company’s longtime president and co-founder, Greg Brockman, resigned — as did three senior OpenAI researchers. And the fallout continues.

TIP TECHCRUNCH

Do you work at OpenAI and know more about Sam Altman’s departure? Get in touch with TechCrunch.
It’s a fast-moving situation that we’re still trying to get to the bottom of. No doubt more will become clear as time goes on. To make it easier to follow all that’s happened in the meantime, though, we’ve put together a timeline; we’ll do our best to keep it current.

Timeline of Sam Altman’s firing from OpenAI

November 29

Microsoft gains a board observer

Microsoft will gain representation on the new initial board of directors in the form of a non-voting observer, OpenAI announced. It wasn’t immediately clear who this observer might be — only that they won’t have an official vote in board business.

November 21

Sam Altman, OpenAI reach agreement on return as CEO and ‘initial’ new board

In a sudden late announcement, OpenAI revealed that it and Altman “have reached an agreement in principle” for him to return as the company’s CEO. In addition to Altman’s return, its new “initial” board will include former Salesforce chief executive Bret Taylor, former US Secretary of the Treasury Larry Summers and Quora founder Adam D’Angelo.


Altman also posted about the new deal, giving some insight into the roller coaster that we’ve all been riding since his firing was revealed Friday. In his words, his decision to join Microsoft on Sunday “was the best path for me and the team.” Since then, the new board’s composition and Microsoft’s support appear to have been enough to bring him back to the AI company he co-founded.

Altman and board in talks

OpenAI’s board of directors is reportedly in talks with Sam Altman, ex-Y Combinator president and an OpenAI co-founder, to return to OpenAI as CEO as soon as this week. That’s according to Bloomberg, which in a brief — citing sources close to the matter — said that discussions are happening between Quora CEO Adam D’Angelo, one current member of the OpenAI board, and Altman — and possibly other board members as well.

Board tensions boil over

The New York Times reports that, before his ousting, Sam Altman made a move to push out board member Helen Toner because he thought a paper she had co-written was overly critical of OpenAI. That, among other issues, led to OpenAI’s current predicament. Speaking of, The Times indicates that negotiations to hire Altman back continue — but that one major sticking point remaining is “guardrails” meant to improve Altman’s communication with the board.


—

November 20

Altman joins Microsoft

Sam Altman, Greg Brockman and colleagues announce that they’ll join Microsoft to lead a new AI research team. Nadella leaves the door open to other OpenAI staffers, saying that they’ll be given the resources they need should they choose to join.

Sutskever’s mea culpa

Sutskever publishes a post on X suggesting that he regrets his decision to remove Altman and that he’ll do everything in his power to reinstate Altman as CEO.


Employees threaten to resign

Nearly 500 of OpenAI’s roughly 770 employees — including, remarkably, Sutskever — publish a letter saying that they might quit unless the startup’s board resigns and reappoints the ousted Altman. Later Monday, that number climbed to over 650.

Altman and Brockman considering return

As reported by The Verge, Altman’s move to Microsoft isn’t a done deal — and both Altman and Brockman are still open to returning to OpenAI. That is, if the remaining board members who initially fired him step aside.

OpenAI board considers merger

OpenAI’s board of directors approached Dario Amodei, the co-founder and CEO of rival large-language model developer Anthropic, about a potential merger of the two companies, The Information reports. The approach was part of an effort by OpenAI to persuade Amodei to replace Altman as CEO — but Amodei quickly turned down the CEO offer.

—

November 19

Altman to meet at OpenAI HQ

According to The Information, Altman is expected to meet at OpenAI’s San Francisco headquarters as executives at OpenAI push to have him reinstated as CEO. Brockman was invited to join — but it’s unclear whether he’ll take execs up on that invitation.

Board negotiations hit a snag

Bloomberg reports that Lightcap and Murati, among others, are pushing the board to reinstate Altman. But unsurprisingly, the directors are resisting. As of midday Sunday, the board hadn’t resigned out of concern over who could replace them, and were vetting candidates. One possible new addition could be Salesforce co-CEO Bret Taylor.

Altman out, Shear in

Altman won’t be returning as CEO, according to a report in The Information citing an internal memo sent by Sutskever. As the search for a new permanent CEO continues, OpenAI has appointed Emmett Shear, the co-founder of video streaming site Twitch, as interim CEO — replacing Murati.

—

November 18

“Not … in response to malfeasance”

In an internal memo obtained by Axios sent Saturday morning, OpenAI COO Brad Lightcap said yesterday’s announcement “took [the management team] by surprise” and that management had had “multiple conversations with the board to try to better understand the reasons and process behind their decision.” Discussions were ongoing as of Saturday morning, per the memo.

“We can say definitively that the board’s decision was not made in response to malfeasance or anything related to our financial, business, safety, or security/privacy practices,” Lightcap added. “This was a breakdown in communication between Sam and the board … We still share your concerns about how the process has been handled, are working to resolve the situation, and will provide updates as we’re able.”

OpenAI’s funding in jeopardy

The planned sale of OpenAI employee shares that would value the startup at about $86 billion could be in jeopardy. The Information, speaking to three sources formerly with the company, reports that they no longer expect the sale — led by Thrive Capital — to happen, or, if it does, to come with a lesser valuation because of the recent turn of events.

Altman planning new venture

Altman has been telling investors that he’s planning to launch a new venture, according to The Information. Brockman is expected to join the effort — whatever form it takes. (Possibly an AI chip startup.)


Investors pushing for Altman’s return

Investors — furious at the turn of events — are reportedly exerting pressure on OpenAI’s board to reinstate Altman, going so far as to recruit Microsoft. Nadella is said to be sympathetic.

Board agrees to reverse course — in principle

The Verge reports that the board agreed in principle to resign and to allow Altman and Brockman to return. It waffled, however, missing a deadline yesterday by which many OpenAI staffers were set to leave the company. Altman is said to be ambivalent about coming back and asking for “significant” governance changes.

—

November 17

Brockman demoted

Brockman says he got a text from Sutskever shortly after noon on Friday asking for a quick call. After sending a Google Meet link, Brockman was told that he was being removed from the board as chairman “but was vital to the company and would retain his role” as president, and that Altman had been fired.

Altman’s firing publicly announced

OpenAI published a post on its blog announcing the executive shake-up. The company’s management team was aware shortly after.


All-hands meeting

OpenAI held an all-hands meeting Friday afternoon during which Sutskever defended Altman’s ouster. He dismissed suggestions that pushing Altman out amounted to a “hostile takeover,” and claimed that it was necessary to protect OpenAI’s mission of “making AI beneficial to humanity.”

Microsoft releases a statement

Satya Nadella, the CEO of Microsoft, a major investor in — and partner with — OpenAI, published a statement about Altman’s firing:

“As you saw at Microsoft Ignite this week, we’re continuing to rapidly innovate for this era of AI, with over 100 announcements across the full tech stack from AI systems, models and tools in Azure, to Copilot. Most importantly, we’re committed to delivering all of this to our customers while building for the future. We have a long-term agreement with OpenAI with full access to everything we need to deliver on our innovation agenda and an exciting product roadmap; and remain committed to our partnership, and to Mira and the team. Together, we will continue to deliver the meaningful benefits of this technology to the world.”

Brockman quits

Brockman announced his resignation from OpenAI, citing “today’s news.” After sending a memo internally, he published the text on X.


Senior OpenAI researchers resign

Three senior OpenAI researchers resign after Brockman, including the director of research Jakub Pachocki and head of preparedness Aleksander Madry.

—

November 16

Ilya Sutskever schedules call with Altman

According to a post on X (formerly Twitter) from Brockman, Ilya Sutskever, the chief scientist at OpenAI and a co-founder, texted Altman on Thursday evening about scheduling a Friday noon call.


Murati told of Altman’s firing

Brockman alleges that Mira Murati, OpenAI’s CTO and now interim CEO, was informed on Thursday night that Altman would be fired.
"""

In [34]:
# 요약 template
summarize_human_message_prompt = "'{text}' 이 내용 bullet point 사용해서 3줄 요약해줘"
summarize_human_message_prompt_template = HumanMessagePromptTemplate.from_template(summarize_human_message_prompt)
summarize_chat_prompt_template = ChatPromptTemplate.from_messages([summarize_human_message_prompt])

In [35]:
# 번역 template
translate_human_message_prompt = "'{text}' 이 내용 한국어로 번역해줘. 번역된 내용만 응답해줘"
translate_human_message_prompt_template = HumanMessagePromptTemplate.from_template(translate_human_message_prompt)
translate_chat_prompt_template = ChatPromptTemplate.from_messages([translate_human_message_prompt_template])

In [36]:
#summarize
summarize_chain = summarize_chat_prompt_template | llm | StrOutputParser()

# summarize_chain을 실행하면 결과값은 StrOutputParser()로 인해 'str'이 된다.

In [42]:
summarize_chain.invoke(text)

"* OpenAI CEO Sam Altman was abruptly fired by the board, leading to the resignation of president Greg Brockman and several key researchers.  This triggered a chaotic series of events including employee threats to resign, Altman joining Microsoft, and board negotiations for his return.\n*  The board cited communication breakdown as the reason for Altman's dismissal, but internal tensions and power struggles were reported.  Investors pressured for Altman's reinstatement, leading to a new board agreement and his eventual return as CEO.\n* Microsoft played a key role, offering Altman and his team positions and later gaining a board observer seat at OpenAI. The incident highlighted the complex relationship between OpenAI's mission, its leadership, and its powerful investors.\n"

In [43]:
type(summarize_chain.invoke(text))

str

In [ ]:
#

In [44]:
#summarize_chain 결과값인 str을 다시 'text' 이름으로 json형태로 정의하고 이를 translate_chat_prompt_template에 chain 구성
#결과적으로 이중 chain 구조가 된다.

summarize_translate_chain = (
    {"text": summarize_chain}
    | translate_chat_prompt_template
    | llm
    | StrOutputParser() )

In [45]:
result = summarize_translate_chain.invoke({"text": text})

In [46]:
print(result)

* 오픈AI CEO 샘 올트먼이 이사회에 의해 전격 해임되었고, 이로 인해 그렉 브록먼 사장과 여러 핵심 연구원들이 사임했습니다. 주요 투자자인 마이크로소프트는 이 사건 직전에 이사회 참관인 자리를 확보했습니다.

* 올트먼이 다른 오픈AI 직원들과 함께 마이크로소프트에 합류하여 새로운 AI 연구팀을 구성하자, 오픈AI 직원 대다수가 올트먼이 복귀하지 않으면 사임하겠다고 위협했습니다. 그 후 이사회는 그의 복귀를 위한 협상을 시작했습니다.

* 격렬한 협상과 내부 혼란 끝에 올트먼은 브렛 테일러, 래리 서머스, 아담 디앤젤로를 포함한 새로운 이사회와 함께 CEO로 복귀했습니다.



In [12]:
print(result)

- **샘 알트만의 해고와 그 후:** 샘 알트만이 OpenAI의 CEO에서 갑작스럽게 해고되었고, 이로 인해 공동 창립자인 그렉 브록먼과 여러 고위 연구원이 사임했다.
- **직원 및 투자자 반응:** 거의 500명의 OpenAI 직원들이 알트만이 복직되지 않으면 사임하겠다고 위협했으며, 투자자들은 이사회를 압박하여 그의 복귀를 요청했다.
- **협상과 복귀:** 알트만과 OpenAI는 새로운 이사회 구성과 함께 그가 CEO로 복귀하는 협정에 도달했으며, 이사회에는 전 Salesforce CEO 브렛 테일러와 같은 인물이 포함되었다.


## Built-in Chains

- 현재 langchain v0.2에는 정리된 페이지가 없음
- https://python.langchain.com/v0.1/docs/modules/chains/

- 랭체인에서 미리 만들어 논 Multichain들을 사용할수 있다.
- 관련 url : https://python.langchain.com/v0.1/docs/modules/chains/

![sql](./res/sql_usecase.png)

In [13]:
#!pip install langchain-experimental

### 실습준비

* [this file](https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql) 이 파일을 data 디렉토리 하위에 저장 `data/Chinook_Sqlite.sql`
* `sqlite3 Chinook.db` 실행
* `.read Chinook_Sqlite.sql` 실행
* `SELECT * FROM Artist LIMIT 10;` 실행 으로 테스트
* `.exit` 또는 `.quit`으로 sqlite3 종료 

Mac의 경우 기본적으로 sqlite3가 설치되어 있고, 만약 설치가 되어있지 않은 경우 공식 홈페이지에서 설치 [link](https://www.sqlite.org/index.html)

In [47]:
from langchain_community.utilities import SQLDatabase

In [48]:
db = SQLDatabase.from_uri("sqlite:///data/Chinook.db")

OperationalError: (sqlite3.OperationalError) unable to open database file
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [16]:
db.get_usable_table_names()

['Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track']

In [17]:
print(db.get_table_info())


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Empl

### create_sql_query_chain - SQL 만들기

In [18]:
from langchain.chains import create_sql_query_chain

In [19]:
sql_chain = create_sql_query_chain(llm, db)

In [20]:
response = sql_chain.invoke({"question": "How many employees are there"})
response

'```sql\nSQLQuery: SELECT COUNT(*) AS "EmployeeCount" FROM "Employee"\n```'

### SQLDatabaseChain - DB에서 정보 추출하기

In [21]:
from langchain_experimental.sql import SQLDatabaseChain

In [30]:
db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True)

In [31]:
db_chain.run("고용된 인원은 몇명이야?")



> Entering new SQLDatabaseChain chain...
고용된 인원은 몇명이야?
SQLQuery:SQLQuery: SELECT COUNT(*) AS "EmployeeCount" FROM "Employee"
SQLResult: [(8,)]
Answer:8명이 고용되어 있습니다.
> Finished chain.


'8명이 고용되어 있습니다.'

In [34]:
db_chain.run("나라별 총 매출을 나열하십시오. 어느 나라의 고객이 가장 많이 소비했어?")



> Entering new SQLDatabaseChain chain...
나라별 총 매출을 나열하십시오. 어느 나라의 고객이 가장 많이 소비했어?
SQLQuery:SQLQuery: SELECT "BillingCountry", SUM("Total") AS "TotalSales" FROM "Invoice" GROUP BY "BillingCountry" ORDER BY "TotalSales" DESC LIMIT 5;
SQLResult: [('USA', 523.06), ('Canada', 303.96), ('France', 195.1), ('Brazil', 190.1), ('Germany', 156.48)]
Answer:USA의 고객이 가장 많이 소비했습니다.
> Finished chain.


'USA의 고객이 가장 많이 소비했습니다.'

### create_retrieval_chain - RAG를 위한 Chain

In [35]:
#!pip install langchainhub

In [7]:
from langchain.text_splitter import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

In [8]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain import hub

In [17]:
#Gemini 임베딩 모델
from langchain_google_genai import GoogleGenerativeAIEmbeddings


In [9]:
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro")

In [39]:
#llm = ChatOpenAI(model="gpt-4-1106-preview")

In [10]:
#lanchain_hub에서 미리 만들어 논 ChatPromtTemplate를 받아 온다.
retrieval_qa_chat_prompt = hub.pull("langchain-ai/retrieval-qa-chat")

In [11]:
#context(문서)와 input(질문)이 있는 template
retrieval_qa_chat_prompt

ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMes

In [12]:
loader = PyPDFLoader("./data/소개팅_나무위키.pdf")

In [13]:
pages = loader.load_and_split()

In [14]:
len(pages)

22

In [15]:
text_splitter = CharacterTextSplitter(separator='\n', chunk_size=250, chunk_overlap=0)
texts = text_splitter.split_documents(pages)

In [16]:
len(texts)

224

In [18]:
#open AI embedding
#embeddings = OpenAIEmbeddings() 
#Gemini embedding
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
db = FAISS.from_documents(texts, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 10})

In [19]:
combine_docs_chain = create_stuff_documents_chain(
    llm, retrieval_qa_chat_prompt
)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [20]:
retrieval_chain.invoke({"input": "주선자의 역할에 대해 알려줘"})

{'input': '주선자의 역할에 대해 알려줘',
 'context': [Document(metadata={'source': './data/소개팅_나무위키.pdf', 'page': 3}, page_content='케이스가  그나마  괜찮은  케이스라  볼  수  있다 .\n소개를  해  주는  입장에서도  사실  어느  정도  부족한  점을  감안해서  비슷한  급의  사람을  소개해준다고  보면  된\n다. 외모든 , 성격이든 , 종교적인  차이든  소개해  주는  쪽에서는  이런  문제들을  대체로  숨기니까  소개팅  하는  남'),
  Document(metadata={'source': './data/소개팅_나무위키.pdf', 'page': 14}, page_content='뷔페식  샐러드바의  경우도  위와  비슷한  이유로  만남  초기엔  그다지  좋지  않다 .\n너무 자극적이거나  매운  음식은  피하자 .\n떡볶이나  매운  닭발이나  마라탕  같은  땀이  뻘뻘  나는  음식은  그  먹는  모습이  아름답지도  않거니와  서로에\n게 땀냄새를  풍기게  될  수  있으니  피하자 .'),
  Document(metadata={'source': './data/소개팅_나무위키.pdf', 'page': 7}, page_content='런 경우엔  친밀한  사람들은  잘  알고있는  본인의  결함이  단지  잠시  걸러지지  못했을  뿐일  가능성이  높다 . 평소\n친한 관계일수록  조금이나마  상대적으로  막  대하는  경향이  있다면 , 설령  소개팅이  들어오더라도  친밀한  관계\n보다는  먼  관계에서  먼저  들어오는  경험을  하게  된다 .'),
  Document(metadata={'source': './data/소개팅_나무위키.pdf', 'page': 4}, page_content='이 오르면  오를수록  원서를  내고자  하는  대학교  기준이 현실적이  되어가는  자신을  발견하듯이  연애  경험\n이 적고 , 현실  감